In [1]:
import pandas as pd
import requests
from datetime import datetime
import socket
import uuid
import time
import requests 
import json
import talib
import http
import ssl
import os

In [2]:
from SmartApi import SmartConnect 
import pyotp
from logzero import logger

In [3]:
# Static values
user_type = "USER"
source_id = "WEB"  
api_key = os.environ["ANG_ONE_KEY"]  
client_code = os.environ["CLIENTCODE"]
password = os.environ["PASSWORD"]
bot_token =  os.environ["BOT_TOKEN"]
test_mode = os.environ["TEST_MODE"].lower() == 'true'
# chat_id = os.environ["CHAT_ID"]
account_sid = os.environ["ACCOUNT_SID"]
window = 365

todays_date = (datetime.today() - pd.DateOffset(days=0)).strftime("%Y-%m-%d")
window_date = (datetime.today() - pd.DateOffset(days=window)).strftime("%Y-%m-%d")

In [4]:
# import requests

def sendWSP(message, apikey,gid=0):
    url = "https://whin2.p.rapidapi.com/send"
    headers = {
	"content-type": "application/json",
	"X-RapidAPI-Key": apikey,
	"X-RapidAPI-Host": "whin2.p.rapidapi.com"}
    try:
        if gid==0:
            return requests.request("POST", url, json=message, headers=headers)
        else: 
            url = "https://whin2.p.rapidapi.com/send2group"
            querystring = {"gid":gid}
            return requests.request("POST", url, json=message, headers=headers, params=querystring) 
    except requests.ConnectionError:
        return("Error: Connection Error")

# Testing Section
msg1 = {"text":"hello there"}
msg2 = {"text":"this is a group message"}

myapikey = "4bd2989815mshc47592a20bf567ep10671ajsn0552a7f118ae"
mygroup = "your_wsp_group_id"

sendWSP(msg1,myapikey)
# sendWSP(msg2, myapikey,mygroup)

<Response [201]>

In [5]:
local_ip = socket.gethostbyname(socket.gethostname())
smartApi = SmartConnect(api_key)

try:
    token = os.environ["TOTP_TOKEN"]
    totp = pyotp.TOTP(token).now()
except Exception as e:
    logger.error("Invalid Token: The provided token is not valid.")
    raise e


# Get Public IP
public_ip = requests.get('https://api.ipify.org').text

# Get MAC Address
mac_address = ':'.join(['{:02x}'.format((uuid.getnode() >> ele) & 0xff)
                        for ele in range(0,8*6,8)][::-1])

# Change clientcode, password, totp
payload = '''{\n\"clientcode\":\"'''+str(client_code)+'''\"
         ,\n\"password\":\"'''+str(password)+'''\"\n
		,\n\"totp\":\"'''+str(totp)+'''\"\n
    ,\n\"state\":\"Active\"\n}'''

headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    "X-UserType": user_type,
    "X-SourceID": source_id,
    "X-ClientLocalIP": local_ip,
    "X-ClientPublicIP": public_ip,
    "X-MACAddress": mac_address,
    'X-PrivateKey': api_key 
}


context = ssl._create_unverified_context()

conn = http.client.HTTPSConnection(
    "apiconnect.angelone.in", context=context
    )

conn.request("POST", "/rest/auth/angelbroking/user/v1/loginByPassword", payload, headers)

res = conn.getresponse()
data = res.read()
data = data.decode("utf-8")

[I 251110 09:29:45 smartConnect:121] in pool


In [6]:
temp = json.loads(data)
jwtToken = temp["data"]["jwtToken"]
print(jwtToken)


local_ip = socket.gethostbyname(socket.gethostname())
public_ip = requests.get('https://api.ipify.org').text
mac_address = ':'.join(['{:02x}'.format((uuid.getnode() >> ele) & 0xff)
                        for ele in range(0,8*6,8)][::-1])
authToken = f'Bearer {jwtToken}'


headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    "X-UserType": user_type,
    "X-SourceID": source_id,
    "X-ClientLocalIP": local_ip,
    "X-ClientPublicIP": public_ip,
    "X-MACAddress": mac_address,
    'X-PrivateKey': api_key,
    'Authorization': authToken ,
}


eyJhbGciOiJIUzUxMiJ9.eyJ1c2VybmFtZSI6IkJHQkcxMTQ0Iiwicm9sZXMiOjAsInVzZXJ0eXBlIjoiVVNFUiIsInRva2VuIjoiZXlKaGJHY2lPaUpTVXpJMU5pSXNJblI1Y0NJNklrcFhWQ0o5LmV5SjFjMlZ5WDNSNWNHVWlPaUpqYkdsbGJuUWlMQ0owYjJ0bGJsOTBlWEJsSWpvaWRISmhaR1ZmWVdOalpYTnpYM1J2YTJWdUlpd2laMjFmYVdRaU9qRXhMQ0p6YjNWeVkyVWlPaUl6SWl3aVpHVjJhV05sWDJsa0lqb2lZMlZrWkRreU9XWXRaV1ZsWkMwek1ERmlMV0k1TldVdE9EZ3lZVEk1TkdVM01EQTFJaXdpYTJsa0lqb2lkSEpoWkdWZmEyVjVYM1l5SWl3aWIyMXVaVzFoYm1GblpYSnBaQ0k2TVRFc0luQnliMlIxWTNSeklqcDdJbVJsYldGMElqcDdJbk4wWVhSMWN5STZJbUZqZEdsMlpTSjlMQ0p0WmlJNmV5SnpkR0YwZFhNaU9pSmhZM1JwZG1VaWZYMHNJbWx6Y3lJNkluUnlZV1JsWDJ4dloybHVYM05sY25acFkyVWlMQ0p6ZFdJaU9pSkNSMEpITVRFME5DSXNJbVY0Y0NJNk1UYzJNamcxTXpNNE5Td2libUptSWpveE56WXlOelkyT0RBMUxDSnBZWFFpT2pFM05qSTNOalk0TURVc0ltcDBhU0k2SWpFd1lqUTNabVEwTFdSaE1tSXRORFl5TWkxaE1XVmxMV05qTldSaE0yWTRZMkUyWXlJc0lsUnZhMlZ1SWpvaUluMC5wSnJXVWZBQ0doMjNHazZCcW1oVUlPcE9sQlA4UHNKMXNaRXp2WS1qd3JYcTFxWG9Fa0NidmRjLVdJNUtHeVQ0NDJMV0ZVRWNaR3JpNzllTkF5WHNhR2d4TEFqcjBlTk1Wd283OWpEQ2RrWnlmOW5KUG9jR2Z

In [7]:
main_df = pd.read_csv('Main_df.csv')

In [8]:
# Step 2: Function to fetch daily candle data from API
def fetch_candle_data(symbol,interval='ONE_DAY'):
  
    payload = '''{\r\n     \"exchange\": \"NSE\",\r\n
          \"symboltoken\": \"'''+str(symbol)+'''\",\r\n     \"interval\": \"'''+str(interval)+'''\",\r\n
          \"fromdate\": \"'''+str(window_date)+''' 16:30\",\r\n     \"todate\": \"'''+str(todays_date)+''' 16:30\"\r\n}
    '''    
    conn = http.client.HTTPSConnection("apiconnect.angelone.in", context=context)
    conn.request("POST", "/rest/secure/angelbroking/historical/v1/getCandleData", payload, headers)
    res = conn.getresponse()
    data = res.read()
    data = data.decode("utf-8")
    json_data = json.loads(data)
    json_data = json_data['data']
    
    return json_data
  

# Fuction to identify RSI Breakout
def rsi_trend(rsi_values, setup_rsi):
    if rsi_values[-1] >= setup_rsi and  rsi_values[-2] < setup_rsi:
      return True
    


In [9]:
# Prepare output DataFrame
error_data = []
priority_data = []

print(main_df.columns.tolist())

# Step 5: Process each stock
for _, row in main_df[main_df['priority'] == 2].iterrows():

    time.sleep(0.4)

    priority = row['priority']
    name = row['Symbol']
    token = row['token']
    rsi = row['rsi']
    win_ratio = row['win_ratio']
    try:

        if rsi == None:
            error_data.append({
               'Name': name,
               'Token': token,
               'Setup_RSI': rsi,
               'reasone': 'RSI Not Maintained in file'
             }) 
            continue
   
   
        # Fetch daily data
        daily_json_data = fetch_candle_data(token)
        if daily_json_data == None:
            error_data.append({
               'Name': name,
               'Token': token,
               'Setup_RSI': rsi,
               'reasone': 'Error while fetching data'
             })    
            continue
         
        df = pd.DataFrame(daily_json_data, columns=["Date", "Open", "High", "Low", "Close", "Volume"])
        
        # break

        # Convert 'Date' column to datetime
        df['Date'] = pd.to_datetime(df['Date'])

        # Sort data by date in ascending order
        df = df.sort_values('Date').reset_index(drop=True)

        df['RSI_14'] = talib.RSI(df['Close'], timeperiod=14)
        df.set_index('Date', inplace=True)
                
        last_2_rsi_daily = df['RSI_14'].dropna().tail(2).values


        if len(last_2_rsi_daily) < 2:
             error_data.append({
               'Name': name,
               'Token': token,
               'Setup_RSI': rsi,
               'reasone': 'Not enough RSI data'
             })
             continue
        daily_rsi = last_2_rsi_daily[-1]

        last_dats = daily_json_data[-1][0].split('T')[0]
        
        if last_dats != todays_date:
             error_data.append({
               'Name': name,
               'Token': token,
               'Setup_RSI': rsi,
               'reasone': f'Date from API {last_dats}, processing date {todays_date}'
             })
             continue


        if priority == 2:
            if rsi_trend(last_2_rsi_daily, rsi):
                priority_data.append({
                    'Name': name,
                    'Token': token,
                    'Setup_RSI': rsi,
                    'Daily_RSI': daily_rsi,
                    'yesterday_RSI': last_2_rsi_daily[-2],
                    'win_ratio': win_ratio,

                })

    except Exception as e:
        error_data.append({
            'Name': name,
            'Token': token,
            'Setup_RSI': rsi,
            'reasone': str(e)
        })
        print(f"Error processing {name}: {e}")

['rsi', 'symbol', 'win_ratio', 'priority', 'Symbol', 'token']


In [10]:
print(main_df)

    rsi      symbol win_ratio  priority      Symbol  token
0    42      360ONE    62.50%         1      360ONE  13062
1    50         ACI    69.44%         0         ACI  12030
2    46       ALKEM    63.01%         2       ALKEM  11703
3    56  ANANDRATHI    68.63%         0  ANANDRATHI   7145
4    74      ANURAS    87.50%         0      ANURAS   2829
..  ...         ...       ...       ...         ...    ...
68   51  TORNTPHARM    44.30%         0  TORNTPHARM   3518
69   55    TVSMOTOR    65.75%         1    TVSMOTOR   8479
70   51  ULTRACEMCO    62.07%         1  ULTRACEMCO  11532
71   52    UNITDSPR    70.31%         0    UNITDSPR  10447
72   57   ZYDUSLIFE    67.74%         0   ZYDUSLIFE   7929

[73 rows x 6 columns]


In [11]:
import requests
# import urllib.parse

# BOT_TOKEN = "8446280700:AAEVJcAw73988-gAx8kJF1TKFMLwHVCM-gs"


TEST_ID = "529251493"
if test_mode:
    CHAT_ID = TEST_ID
else:
    CHAT_ID = "-1003139839259"


    
def format_whatsapp_report(data ,name):
    
    lines = [f"📊 <b>{name}</b>"]
    
    if len(data) == 0:
        lines.append( "\n🔹 <b>No Stocks</b>" )
    else:
        for index,item in enumerate(data):
            lines.append(
                f"\n🔹 <b>{index+1} {item['Name']}</b>"
                f"\n   <b>Todays RSI :</b> {item['Daily_RSI']:.2f}"
                f"\n   <b>Yesterdays RSI :</b> {item['yesterday_RSI']:.2f}"
                f"\n   <b>Standard RSI :</b> {item['Setup_RSI']:.2f}"
            )      
           
    # return urllib.parse.quote_plus( "\n".join(lines) )
    return "\n".join(lines) 

def format_whatsapp_error(data ,name):
    
    lines = [f"📊 <b>{name}</b>"]
    
    if len(data) == 0:
        lines.append( "\n🔹 <b>No Stocks</b>" )
    else:
        for index,item in enumerate(data):
            lines.append(
                f"\n🔹 <b>{index+1} {item['Name']}</b>"
                f"\n   <b>Token: </b> {item['Token']}"
                f"\n   <b>Standard RSI: </b> {item['Setup_RSI']:.2f}"
                f"\n   <b>Reasone: </b> {item['reasone']}"
            )      
           
    return "\n".join(lines)


In [12]:
print(priority_data)

[{'Name': 'MARUTI', 'Token': 12587, 'Setup_RSI': 38, 'Daily_RSI': np.float64(40.72406232627147), 'yesterday_RSI': np.float64(37.09202765433505), 'win_ratio': '90.00%'}]


In [13]:
# Telegram Message trigger logic
if priority_data is not None:
    msg = f"📊 <b>Live Stock: {todays_date}</b>"
    # requests.get(f"https://api.telegram.org/bot{bot_token}/sendMessage",
    #             params={"chat_id": CHAT_ID, "text": msg, "parse_mode": "HTML"})

    msg_p1 = format_whatsapp_report(priority_data,'Priority Stocks')
    # requests.get(f"https://api.telegram.org/bot{bot_token}/sendMessage",
    #             params={"chat_id": CHAT_ID, "text": msg_p1, "parse_mode": "HTML"})


In [14]:

# if error_data is not None:
#     error_msg = format_whatsapp_error(error_data,'Error Stocks')
#     requests.get(f"https://api.telegram.org/bot{bot_token}/sendMessage",
#                 params={"chat_id": TEST_ID, "text": error_msg, "parse_mode": "HTML"})


In [15]:
print(error_data)

[{'Name': 'SBFC', 'Token': 18030, 'Setup_RSI': 37, 'reasone': 'Error while fetching data'}, {'Name': 'SOLARINDS', 'Token': 13332, 'Setup_RSI': 61, 'reasone': 'Error while fetching data'}, {'Name': 'THERMAX', 'Token': 3475, 'Setup_RSI': 45, 'reasone': 'Error while fetching data'}]


In [16]:
# # Create Files For the output

# error_df = pd.DataFrame(error_data)
# error_df.to_csv(f'Daily_Report/Error/RSI-Setup-error-{todays_date}.csv', index=False)
# priority_df = pd.DataFrame(priority_data)
# priority_df.to_csv(f'Daily_Report/Priority/RSI-Setup-Priority-{todays_date}.csv', index=False)

In [17]:
# !pip install twilio


In [ ]:
# import twilio.rest
# import http.client

In [ ]:
# from twilio.rest import Client
# import urllib.parse

# # Your Twilio account credentials
# auth_token = '47718966186ae707e5f663daf1a2c714'

# # Create a Twilio client
# client = Client(account_sid, auth_token)

# msg_p1 = msg_p1.replace('<b>', '*').replace('</b>', '*')
# # msg_p1 = urllib.parse.quote_plus( "\n".join(msg_p1) )
# print(msg_p1)

# # def send_whatsapp_message(to_number, message_body):
# # Send a WhatsApp message
# message = client.messages.create(
#     from_='whatsapp:+14155238886',  # Twilio sandbox number (from console)+14155238886
#     body= msg_p1, #'Hello! This is a test message from Twilio WhatsApp API 🚀',
#     to='whatsapp:+919004841091'     # Replace with recipient's full number including country code
# )

# print(f"✅ Message sent! SID: {message}")
# msg = client.messages(message.sid).fetch()
# print(msg.status)

📊 *Priority Stocks*

🔹 *1 MARUTI*
   *Todays RSI :* 40.72
   *Yesterdays RSI :* 37.09
   *Standard RSI :* 38.00


TwilioRestException: HTTP 401 error: Unable to create record: Authenticate

In [ ]:
# import requests

# # ==== Replace these with your actual values ====
# ACCESS_TOKEN = "YOUR_META_PERMANENT_ACCESS_TOKEN"
# PHONE_NUMBER_ID = "YOUR_PHONE_NUMBER_ID"  # e.g. 123456789012345
# GROUP_ID = "120363025556871234@g.us"      # found via webhook or logs

# # ===============================================

# url = f"https://graph.facebook.com/v20.0/{PHONE_NUMBER_ID}/messages"

# headers = {
#     "Authorization": f"Bearer {ACCESS_TOKEN}",
#     "Content-Type": "application/json"
# }

# payload = {
#     "messaging_product": "whatsapp",
#     "to": GROUP_ID,
#     "type": "text",
#     "text": {
#         "body": "🚀 Hello, WhatsApp group! This is an automated message from the WhatsApp Cloud API."
#     }
# }

# response = requests.post(url, headers=headers, json=payload)

# # === Output for debugging ===
# print("Status code:", response.status_code)
# print("Response:", response.text)
